# Skyra TSER Recon

### 1. Import Libraries

In [1]:
import mapvbvd
import numpy as np 
import matplotlib.pyplot as plt
%matplotlib qt

/home/hans/Documents/TSER/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### 2. Load dataset

In [80]:
def load_TSER(file, points, echoes, channels = 15):
    twixObj = mapvbvd.mapVBVD(file)
    twixObj[1].image.squeeze = True #remove extra dimensions
    twixObj[1].image.flagRemoveOS = False #no readout oversampling
    data = twixObj[1].image[''].reshape(points,channels,-1, 2, echoes) # arrange data into (points, channels, slices, phase cycle, echoes)
    data_pc = data[:,:,:,0] + data[:,:,:,1] # add phase cycle steps to remove ringing

    # data_pc is phase cycled data with shape (points, channels, slices, echoes)
    return data_pc

data = load_TSER('Data/CMIF/Oct5/meas_MID00101_FID02502_400a_std_200.dat', 128, 400)

pymapVBVD version 0.7.0
Software version: VD


### 3. Fourier Transform to get Image

In [81]:
data_ft = np.fft.fftshift(np.fft.fft(np.fft.fftshift(data[3:-3,:,:,:], axes=(0)), axis=0), axes=(0))

### 4. Plot 3D data for one slice

In [84]:
fig, ax = plt.subplots(subplot_kw={"projection": "3d"})
ax.view_init(elev=30, azim=-26, roll=0)

distance = np.linspace(-54,54,np.shape(data_ft)[0])
colors = plt.cm.viridis(np.linspace(0, 1, 400))

for i in range(400):
    ax.plot(i, distance, np.abs(data_ft[:,0,0,i]),color=colors[i], lw=1) #every point, channel 6, slice 0, echo 'i'

plt.show()

### 5. Plot Slice Cross Section

In [79]:
plt.figure()
plt.plot(np.abs(data_ft[:,0,:,5])) #Every point, channel 6, slice 0, echo 5
plt.xlabel("Index")
plt.ylabel("Amplitude")
plt.show()

### 5.5 Plot Plate Map

In [92]:
fig, ax = plt.subplots(1,1)
ax.imshow(np.abs(data_ft[:,0,:,250]), aspect='auto') #Every point, channel 6, slice 0, echo 5
plt.xlabel("Index")
plt.ylabel("Amplitude")
plt.show()

### 6. Plot Decay for One Point

In [85]:
point = 21
echoes = 400
te = 10e-3
plt.figure()
time = np.arange(echoes)*te
plt.plot(time, np.abs(data_ft[40,6,0,:])) #Point 40, channel 6, slice 0, every echo
plt.xlabel("Distance (mm)")
plt.ylabel("Amplitude")
plt.show()